## Investigating model complexity, degrees of freedom to provide a judicious choice of hyperparameters for platonic representation hypothesis testing

In [ ]:
import sys
sys.path.append('../../../')
from polygene.model.model import load_trained_model, Polygene
import transformers
import numpy as np, pandas as pd 

# we embed cell states into a smooth manifold of attractor bassins, we have emergent phenomenology of sem, trajectorial inference. 
# tell Elazer. wrapping up promising -  Hey Elazer- I thought I should let you know I wont be in lab today, you might be expecting an update, i am finishing some results and writing
# for any other reasons 
# really looking forward to our next meeting.

# for training stability we add a learnable parameter of temperature which gives us control on how tight the simplexes are populated. 
model, tokenizer = load_trained_model('../../../runs/gesam_polygene_run_4/', checkpoint_n=-1)

def count_model_parameters(model):
    
    # inference weights, model weights
    transformer_weights = sum(parameter.numel() for name, parameter in model.named_parameters() if not 'embedding' in name) 

    # vocabulary included, total DoF
    total_weights = sum(parameter.numel() for name, parameter in model.named_parameters())
    return transformer_weights, total_weights


grid_search_parameters = {
    "n_layers": [1, 3, 6],
    "dim": [96, 144, 240], # main toggle or knob to increase model complexity
    "seeds": [2, 3], # control the random weight initialisation and shuffle of the training data
}

reference_model_params = {"seeds": 3, "n_layers": 6,  "dim": 240}

models = []
for hyperparameter in grid_search_parameters:
    for value in grid_search_parameters[hyperparameter]:
        model_config = transformers.AutoConfig.from_pretrained("../../model/polygene_architecture.json",attn_implementation="flash_attention_2")
        seed = reference_model_params["seeds"] if not hyperparameter == "seeds" else value
        model_config.n_layers = reference_model_params["n_layers"] if not hyperparameter == "n_layers" else value
        model_config.dim = reference_model_params["dim"]  if not hyperparameter == "dim" else value
        
        model_config.n_heads = 6
        model_config.hidden_dim = model_config.dim * 2

        model_config.vocab_size = tokenizer.vocab_size 
        model_config.type_vocab_size = tokenizer.type_vocab_size
        model_config.pad_token_id = tokenizer.convert_tokens_to_ids(tokenizer.pad_token)

        model = Polygene._from_config(model_config, ** {"attn_implementation": "flash_attention_2"})
        forward_params, total_params = count_model_parameters(model)
        models .append( {"name": f"polygene_unit_sphere_seed{seed}_nlayer{model_config.n_layers}_dim{model.config.dim}" 
                  if not all(reference_model_params[key] == val for key, val in zip(reference_model_params.keys(), [seed, model_config.n_layers, model.config.dim])) else "reference_model",
                "encoder_params": forward_params, "total_params": total_params})
df = pd.DataFrame.from_dict(models).drop_duplicates('name')

Some weights of Polygene were not initialized from the model checkpoint at ../../../runs/gesam_polygene_run_4/checkpoint-2300000 and are newly initialized: ['temperature']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [6]:
units = ['', 'K', 'M', 'B', 'T']
df[['encoder_param_h', 'total_params_h']] = df[['encoder_params', 'total_params']].applymap(lambda v:f"{v / 10**(int(np.log10(v) // 3)*3):.1f}{units[int(np.log10(v) // 3)]}")
df.sort_values('total_params')

,name,encoder_params,total_params,encoder_param_h,total_params_h
3,polygene_unit_sphere_seed3_nlayer6_dim96,449984,6181088,450.0K,6.2M
4,polygene_unit_sphere_seed3_nlayer6_dim144,1006208,9602864,1.0M,9.6M
0,polygene_unit_sphere_seed3_nlayer1_dim240,465008,14792768,465.0K,14.8M
1,polygene_unit_sphere_seed3_nlayer3_dim240,1391888,15719648,1.4M,15.7M
2,reference_model,2782208,17109968,2.8M,17.1M
6,polygene_unit_sphere_seed1_nlayer6_dim240,2782208,17109968,2.8M,17.1M
7,polygene_unit_sphere_seed2_nlayer6_dim240,2782208,17109968,2.8M,17.1M
